# Questão 1 — Segmentação de clientes (K-means)

**Por que K-means?** O dataset não tem coluna de rótulo e o objetivo é *descobrir* perfis de clientes automaticamente → problema **não supervisionado de agrupamento**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # reprodutibilidade

# 1. ler dataset csv
X = np.genfromtxt("dataset_marketing.csv", delimiter=",", skip_header=1)

## 1. Conhecendo os dados
Antes de escolher k, confiro o formato, as escalas e o aspecto dos dados.

In [ ]:
print("shape:", X.shape)
print("mínimo por coluna :", X.min(axis=0).round(2))
print("máximo por coluna :", X.max(axis=0).round(2))
print("média por coluna  :", X.mean(axis=0).round(2))
print("desvio por coluna :", X.std(axis=0).round(2))
print("valores ausentes  :", np.isnan(X).sum())

plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.7)
plt.title("Dados brutos (sem agrupar)")
plt.xlabel("Renda Anual (valor relativo)")
plt.ylabel("Score de Gastos (valor relativo)")
plt.grid(True)
plt.show()

**Observações:** as médias ficam perto de zero e há valores negativos — portanto **não** estão na escala original (renda em milhares; score de 1 a 100). São valores relativos (dados centralizados), então "alto"/"baixo" significa acima/abaixo da média. As duas colunas têm escalas parecidas (ver desvios), então não é necessário normalizar.

## 2. Implementação do K-means (mesmo código da aula 12)

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # reprodutibilidade

# 1. ler dataset csv
X = np.genfromtxt("dataset_marketing.csv", delimiter=",", skip_header=1)

## 1. Conhecendo os dados
Antes de escolher k, confiro o formato, as escalas e o aspecto dos dados.

## 3. Escolha de k — método do cotovelo
A inércia (soma dos quadrados das distâncias de cada ponto ao seu centróide) sempre cai quando k aumenta; procuro o k em que a queda deixa de ser grande. Como a inicialização é aleatória, para cada k rodo várias vezes e guardo a **menor** inércia.

In [ ]:
def inertia(X, centroids, labels):
    return sum(np.sum((X[labels == j] - centroids[j]) ** 2) for j in range(len(centroids)))

def melhor_kmeans(X, k, n_init=10):
    """Roda o k-means n_init vezes e devolve a execução de menor inércia."""
    melhor = None
    for _ in range(n_init):
        c, l = kmeans(X, k=k)
        j = inertia(X, c, l)
        if melhor is None or j < melhor[0]:
            melhor = (j, c, l)
    return melhor

ks = range(1, 9)
inercias = []
for k in ks:
    j, _, _ = melhor_kmeans(X, k)
    inercias.append(j)
    print(f"k={k}  inércia={j:10.1f}")

plt.figure(figsize=(8, 5))
plt.plot(list(ks), inercias, marker="o")
plt.xlabel("k (número de clusters)")
plt.ylabel("Inércia")
plt.title("Método do cotovelo")
plt.grid(True)
plt.show()

## 4. Estabilidade da inicialização
Uma única execução pode cair num mínimo local ruim. Rodo várias vezes (1 inicialização cada) para o k escolhido e comparo.

In [ ]:
K = 3  # k escolhido no cotovelo (justificativa na conclusão)

resultados = []
for semente in range(10):
    np.random.seed(semente)
    c, l = kmeans(X, k=K)
    resultados.append(inertia(X, c, l))
    print(f"semente {semente}: inércia = {resultados[-1]:.1f}")

print("\nmenor inércia :", round(min(resultados), 1))
print("execuções que ficaram na menor inércia:", sum(np.isclose(r, min(resultados), rtol=1e-3) for r in resultados), "de", len(resultados))

## 5. Resultado final (melhor execução)

In [ ]:
inercia_final, centroids, labels = melhor_kmeans(X, K, n_init=20)
print(f"inércia final: {inercia_final:.1f}")

# ordena os clusters pela renda do centróide, só para a legenda ficar sempre igual
ordem = np.argsort(centroids[:, 0])
centroids = centroids[ordem]
labels = np.argsort(ordem)[labels.astype(int)]

plt.figure(figsize=(10, 6))
for cluster_id in np.unique(labels):
    plt.scatter(X[labels == cluster_id, 0], X[labels == cluster_id, 1], alpha=0.7, label=f"Cluster {int(cluster_id)}")
plt.scatter(centroids[:, 0], centroids[:, 1], s=180, c="black", marker="X", label="Centróides")
plt.title("Segmentação de Clientes com K-Means")
plt.xlabel("Renda Anual (valor relativo)")
plt.ylabel("Score de Gastos (valor relativo)")
plt.legend()
plt.grid(True)
plt.show()

print("Centróides (renda, score):")
for i, c in enumerate(centroids):
    print(f"  cluster {i}: renda={c[0]:6.2f}  score={c[1]:6.2f}  clientes={int(np.sum(labels == i))}")
print("total de clientes:", int(sum(np.sum(labels == i) for i in range(K))))

## Conclusão

**Algoritmo.** O dataset não tem rótulo e o objetivo é descobrir perfis de clientes, então usei o **K-means** (agrupamento não supervisionado). Por isso não há "acurácia": a qualidade é avaliada pela inércia, pelo gráfico dos clusters e pela interpretação dos centróides.

**Escolha de k = 3.** No gráfico do cotovelo, a inércia cai muito de k=1 (≈ 21 307) para k=2 (≈ 6 555) e para k=3 (≈ 1 275). A partir de k=4 as quedas são pequenas (≈ 1 116, 959, 814...) e a curva "achata". O cotovelo está em **k = 3**, o que também bate com as três nuvens visíveis no gráfico dos dados brutos. Cada cluster ficou com 100 clientes (total 300).

**Estabilidade.** Em 10 execuções com sementes diferentes, 7 chegaram à menor inércia (≈ 1 275) e 3 ficaram presas num mínimo local ruim (≈ 6 400, dois centróides dividindo uma nuvem). Por isso o resultado final usa a **melhor entre várias inicializações** (`melhor_kmeans`), e não uma execução isolada.

**Perfis encontrados** (valores relativos: os dados não estão em escala 1–100; negativo = abaixo da média):

| Cluster | Renda (centróide) | Score de gastos (centróide) | Perfil | Ação de campanha sugerida |
|---|---|---|---|---|
| 0 | ≈ −6,9 (baixa) | ≈ −7,0 (baixo) | Renda baixa e gasto baixo | Promoções de entrada, descontos e produtos de baixo custo |
| 1 | ≈ −2,7 (média-baixa) | ≈ +9,1 (muito alto) | Gasta muito em relação à renda | Programa de fidelidade e parcelamento; cuidado com risco de crédito |
| 2 | ≈ +4,8 (alta) | ≈ +2,0 (levemente acima da média) | Renda alta, gasto moderado | Produtos premium e experiências, com potencial para aumentar o gasto |

**Limitações.** O K-means depende da inicialização aleatória (mitigada com várias execuções), exige escolher k (aqui apoiado no cotovelo) e assume grupos compactos; as colunas já têm escalas parecidas, por isso não normalizei.
